# Incident Detection Agent — Training and Evaluation

This notebook contains the complete pipeline used for the traffic incident agent.

**Pipeline:** AI City Challenge Track 4 → YOLO11 vehicle detection → ByteTrack tracking → 5-second traffic features → Random Forest / XGBoost → temporal confirmation → LangChain agent.

The notebook is written for **Google Colab** and assumes the project is stored in:

`/content/drive/MyDrive/5piproject`

A few early experiments from the development phase were removed or merged so the notebook can be read and rerun from top to bottom without duplicated setup cells.


## 1. Colab setup

Mount Drive, install the packages used by the project, and check that the runtime can see a GPU. The video stage is the expensive part, so a GPU runtime is strongly recommended.


In [ ]:
from google.colab import drive

drive.mount('/content/drive')


In [ ]:
%pip install -q ultralytics opencv-python pandas numpy matplotlib scikit-learn xgboost joblib tqdm openai langchain langchain-openai


In [ ]:
from pathlib import Path
import gc
import json
import shutil
import time

import cv2
import joblib
import numpy as np
import pandas as pd
import torch

from ultralytics import YOLO

print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))


## 2. Project paths and dataset metadata

The training split contains the 100 labeled AI City Track 4 videos. The official 150 test videos are not used here because their labels are not public. Our `video_splits.csv` already separates the 100 labeled videos into train, validation, and held-out test groups by **whole video**.


In [ ]:
PROJECT_ROOT = Path('/content/drive/MyDrive/5piproject')

TRAIN_VIDEO_DIR = (
    PROJECT_ROOT
    / 'data/raw/track4/AIC21-Track4-Anomaly-Detection/aic21-track4-train-data'
)
SPLITS_FILE = PROJECT_ROOT / 'data/processed/video_splits.csv'
ANNOTATIONS_FILE = PROJECT_ROOT / 'data/processed/anomaly_intervals.csv'
OUTPUT_DIR = PROJECT_ROOT / 'data/processed/features_per_video'
LOCAL_WORK_DIR = Path('/content/pi_batch')

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
LOCAL_WORK_DIR.mkdir(parents=True, exist_ok=True)

# Feature extraction settings. TARGET_FPS=5 keeps the workload manageable while
# preserving enough temporal information for tracking and motion features.
WINDOW_SECONDS = 5.0
TARGET_FPS = 5.0
CONFIDENCE_THRESHOLD = 0.25
STOPPED_SPEED_THRESHOLD = 0.08
MIN_STOPPED_OBSERVATIONS = 5

# COCO class ids used by YOLO.
CAR, MOTORCYCLE, BUS, TRUCK = 2, 3, 5, 7
VEHICLE_CLASSES = [CAR, MOTORCYCLE, BUS, TRUCK]

splits = pd.read_csv(SPLITS_FILE)
annotations = pd.read_csv(ANNOTATIONS_FILE)

print('Videos in split file:', len(splits))
print('Annotated anomaly intervals:', len(annotations))
print('\nSplit summary:')
display(pd.crosstab(splits['split'], splits['has_anomaly'], margins=True))


## 3. Feature extraction from video

Each video is sampled at about 5 FPS. YOLO detects vehicles and ByteTrack keeps a track id for each vehicle. From the trajectories we build one row every five seconds.

A few details matter here:
- `normalized_speed` is only an **image-motion proxy**. It is not km/h.
- Bounding-box coverage is used as a rough occupancy proxy.
- The stopped-vehicle rule is deliberately simple and is treated as a supporting feature, not a definitive incident detector.
- A window is labeled as an incident when its midpoint falls inside an annotated anomaly interval.


In [ ]:
DEVICE = 0 if torch.cuda.is_available() else 'cpu'
print('YOLO device:', DEVICE)

# Load once and reuse it across the batch.
yolo_model = YOLO('yolo11n.pt')


def get_ground_truth(video_id: int, midpoint: float) -> int:
    """Return 1 when the midpoint falls inside an annotated anomaly interval."""
    video_annotations = annotations[annotations['video_id'] == video_id]
    for _, anomaly in video_annotations.iterrows():
        if float(anomaly['start_sec']) <= midpoint <= float(anomaly['end_sec']):
            return 1
    return 0


def process_video(video_id: int, split_name: str, local_video: Path) -> pd.DataFrame:
    """Run YOLO + ByteTrack on one video and return 5-second feature rows."""
    cap = cv2.VideoCapture(str(local_video))
    if not cap.isOpened():
        raise RuntimeError(f'Cannot open {local_video}')

    original_fps = cap.get(cv2.CAP_PROP_FPS)
    frame_count = cap.get(cv2.CAP_PROP_FRAME_COUNT)
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    cap.release()

    duration = frame_count / original_fps
    vid_stride = max(1, round(original_fps / TARGET_FPS))
    processed_fps = original_fps / vid_stride
    frame_area = width * height

    frame_rows = []
    detection_rows = []

    results = yolo_model.track(
        source=str(local_video),
        tracker='bytetrack.yaml',
        persist=True,
        stream=True,
        vid_stride=vid_stride,
        conf=CONFIDENCE_THRESHOLD,
        classes=VEHICLE_CLASSES,
        device=DEVICE,
        verbose=False,
        save=False,
    )

    for processed_frame, result in enumerate(results):
        source_time = processed_frame / processed_fps
        boxes = result.boxes
        vehicle_count = 0
        bbox_area_sum = 0.0
        confidences = []

        if boxes is not None:
            for box in boxes:
                # Boxes without a track id cannot be used for trajectory features.
                if box.id is None:
                    continue

                track_id = int(box.id[0].item())
                class_id = int(box.cls[0].item())
                confidence = float(box.conf[0].item())
                x1, y1, x2, y2 = box.xyxy[0].cpu().numpy().tolist()

                bbox_width = x2 - x1
                bbox_height = y2 - y1
                bbox_area = bbox_width * bbox_height
                center_x = (x1 + x2) / 2
                center_y = (y1 + y2) / 2

                vehicle_count += 1
                bbox_area_sum += bbox_area
                confidences.append(confidence)

                detection_rows.append({
                    'processed_frame': processed_frame,
                    'source_time_sec': source_time,
                    'track_id': track_id,
                    'class_id': class_id,
                    'confidence': confidence,
                    'center_x': center_x,
                    'center_y': center_y,
                    'bbox_height': bbox_height,
                    'bbox_area': bbox_area,
                })

        frame_rows.append({
            'processed_frame': processed_frame,
            'source_time_sec': source_time,
            'vehicle_count': vehicle_count,
            'bbox_coverage': min(bbox_area_sum / frame_area, 1.0),
            'mean_detection_confidence': float(np.mean(confidences)) if confidences else 0.0,
        })

    frames = pd.DataFrame(frame_rows)
    detections = pd.DataFrame(detection_rows)

    frames['window_start_sec'] = (
        np.floor(frames['source_time_sec'] / WINDOW_SECONDS) * WINDOW_SECONDS
    ).round(3)

    if not detections.empty:
        detections['window_start_sec'] = (
            np.floor(detections['source_time_sec'] / WINDOW_SECONDS) * WINDOW_SECONDS
        ).round(3)

        # Track-to-track movement. Dividing by box height reduces part of the
        # perspective effect, but this value should still be treated as a proxy.
        detections = detections.sort_values(['track_id', 'source_time_sec']).reset_index(drop=True)
        detections['prev_x'] = detections.groupby('track_id')['center_x'].shift(1)
        detections['prev_y'] = detections.groupby('track_id')['center_y'].shift(1)
        detections['prev_time'] = detections.groupby('track_id')['source_time_sec'].shift(1)

        detections['distance_px'] = np.sqrt(
            (detections['center_x'] - detections['prev_x']) ** 2
            + (detections['center_y'] - detections['prev_y']) ** 2
        )
        detections['dt'] = detections['source_time_sec'] - detections['prev_time']
        detections.loc[detections['dt'] <= 0, 'dt'] = np.nan
        detections['speed_px_sec'] = detections['distance_px'] / detections['dt']
        detections['normalized_speed'] = (
            detections['speed_px_sec'] / detections['bbox_height'].clip(lower=1)
        )
        detections.replace([np.inf, -np.inf], np.nan, inplace=True)

        track_window = (
            detections.groupby(['window_start_sec', 'track_id'], as_index=False)
            .agg(
                observations=('track_id', 'size'),
                median_normalized_speed=('normalized_speed', 'median'),
            )
        )
        track_window['potentially_stopped'] = (
            (track_window['observations'] >= MIN_STOPPED_OBSERVATIONS)
            & (track_window['median_normalized_speed'] < STOPPED_SPEED_THRESHOLD)
        )
    else:
        track_window = pd.DataFrame(
            columns=['window_start_sec', 'track_id', 'observations',
                     'median_normalized_speed', 'potentially_stopped']
        )

    feature_rows = []
    for start in np.arange(0, duration, WINDOW_SECONDS):
        start = round(float(start), 3)
        end = min(start + WINDOW_SECONDS, duration)
        frame_group = frames[frames['window_start_sec'] == start]
        detection_group = (
            detections[detections['window_start_sec'] == start]
            if not detections.empty else detections
        )

        if frame_group.empty:
            mean_vehicle_count = 0.0
            coverage = 0.0
            max_coverage = 0.0
            mean_confidence = 0.0
        else:
            mean_vehicle_count = float(frame_group['vehicle_count'].mean())
            coverage = float(frame_group['bbox_coverage'].mean())
            max_coverage = float(frame_group['bbox_coverage'].max())
            mean_confidence = float(frame_group['mean_detection_confidence'].mean())

        if detection_group.empty:
            unique_vehicles = 0
            mean_speed = median_speed = 0.0
            car_unique = motorcycle_unique = bus_unique = truck_unique = 0
        else:
            unique_vehicles = int(detection_group['track_id'].nunique())
            speeds = detection_group['normalized_speed'].dropna()
            mean_speed = float(speeds.mean()) if len(speeds) else 0.0
            median_speed = float(speeds.median()) if len(speeds) else 0.0

            def count_class(class_id):
                return int(
                    detection_group.loc[detection_group['class_id'] == class_id, 'track_id'].nunique()
                )

            car_unique = count_class(CAR)
            motorcycle_unique = count_class(MOTORCYCLE)
            bus_unique = count_class(BUS)
            truck_unique = count_class(TRUCK)

        stopped_info = track_window[track_window['window_start_sec'] == start]
        stopped_count = int(stopped_info['potentially_stopped'].sum())
        stopped_ratio = stopped_count / unique_vehicles if unique_vehicles else 0.0

        feature_rows.append({
            'video_id': int(video_id),
            'split': split_name,
            'window_start_sec': start,
            'window_end_sec': float(end),
            'unique_vehicle_count': unique_vehicles,
            'mean_vehicle_count_per_frame': mean_vehicle_count,
            'mean_normalized_speed': mean_speed,
            'median_normalized_speed': median_speed,
            'stopped_vehicle_count': stopped_count,
            'stopped_ratio': stopped_ratio,
            'bbox_coverage_proxy': coverage,
            'max_bbox_coverage_proxy': max_coverage,
            'mean_detection_confidence': mean_confidence,
            'car_unique': car_unique,
            'motorcycle_unique': motorcycle_unique,
            'bus_unique': bus_unique,
            'truck_unique': truck_unique,
            'label': get_ground_truth(video_id, (start + end) / 2),
        })

    return pd.DataFrame(feature_rows)


### Quick check on two videos

Before launching the full batch, this small test confirms that the code creates the expected number of windows and that the anomaly labels are aligned correctly. In the current dataset, video 1 is normal while video 2 contains a long anomaly interval.


In [ ]:
for _, row in splits.head(2).iterrows():
    video_id = int(row['video_id'])
    split_name = row['split']
    source = TRAIN_VIDEO_DIR / f'{video_id}.mp4'
    local = LOCAL_WORK_DIR / f'{video_id}.mp4'

    shutil.copy2(source, local)
    try:
        test_features = process_video(video_id, split_name, local)
        print(f'Video {video_id}: {len(test_features)} windows, '
              f'{int(test_features["label"].sum())} incident windows')
    finally:
        local.unlink(missing_ok=True)


## 4. Batch feature extraction

Each video is copied from Drive to Colab's local disk before processing because repeated frame reads from Drive are noticeably slower. A CSV is saved after every video, which also acts as a checkpoint: if Colab disconnects, rerunning the cell skips videos that are already finished.


In [ ]:
split_order = {'train': 0, 'val': 1, 'test': 2}
ordered_splits = (
    splits.assign(split_order=splits['split'].map(split_order))
    .sort_values(['split_order', 'video_id'])
)

successful = 0
failed = []

for counter, (_, row) in enumerate(ordered_splits.iterrows(), start=1):
    video_id = int(row['video_id'])
    split_name = row['split']
    output_file = OUTPUT_DIR / f'video_{video_id:03d}.csv'

    if output_file.exists():
        print(f'[{counter}/{len(ordered_splits)}] video {video_id} already done')
        successful += 1
        continue

    source_video = TRAIN_VIDEO_DIR / f'{video_id}.mp4'
    local_video = LOCAL_WORK_DIR / f'{video_id}.mp4'
    started = time.time()

    try:
        shutil.copy2(source_video, local_video)
        video_features = process_video(video_id, split_name, local_video)
        video_features.to_csv(output_file, index=False)

        incident_windows = int(video_features['label'].sum())
        elapsed_min = (time.time() - started) / 60
        print(f'[{counter}/{len(ordered_splits)}] video {video_id}: '
              f'{len(video_features)} windows, {incident_windows} incidents, '
              f'{elapsed_min:.2f} min')
        successful += 1
    except Exception as exc:
        failed.append((video_id, str(exc)))
        print(f'Error on video {video_id}: {exc}')
    finally:
        local_video.unlink(missing_ok=True)
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

print(f'Completed: {successful}/{len(ordered_splits)}')
print('Failed:', failed)


## 5. Merge per-video features

Once all 100 videos are processed, the individual checkpoint files are merged into a single modeling table.


In [ ]:
feature_files = sorted(OUTPUT_DIR.glob('video_*.csv'))
if len(feature_files) != 100:
    print(f'Warning: found {len(feature_files)} feature files, expected 100.')

all_features = pd.concat(
    [pd.read_csv(file) for file in feature_files],
    ignore_index=True,
).sort_values(['video_id', 'window_start_sec']).reset_index(drop=True)

print('Feature files:', len(feature_files))
print('Total 5-second windows:', len(all_features))
print('\nLabel distribution:')
print(all_features['label'].value_counts())
print('\nSplit / label distribution:')
display(pd.crosstab(all_features['split'], all_features['label'], margins=True))

MASTER_FEATURE_FILE = PROJECT_ROOT / 'data/processed/all_traffic_features.csv'
all_features.to_csv(MASTER_FEATURE_FILE, index=False)
print('Saved:', MASTER_FEATURE_FILE)


## 6. Add temporal features

Incidents are temporal events, so the model should see more than a single snapshot. For selected traffic variables we add:
- the change from the previous 5-second window (`delta_...`), and
- a rolling mean over the current and two previous windows (`roll3_...`).


In [ ]:
df = pd.read_csv(MASTER_FEATURE_FILE)
df = df.sort_values(['video_id', 'window_start_sec']).reset_index(drop=True)

TEMPORAL_COLUMNS = [
    'unique_vehicle_count',
    'mean_vehicle_count_per_frame',
    'mean_normalized_speed',
    'median_normalized_speed',
    'stopped_vehicle_count',
    'stopped_ratio',
    'bbox_coverage_proxy',
]

for column in TEMPORAL_COLUMNS:
    df[f'delta_{column}'] = df.groupby('video_id')[column].diff().fillna(0)
    df[f'roll3_{column}'] = (
        df.groupby('video_id')[column]
        .transform(lambda s: s.rolling(3, min_periods=1).mean())
    )

TEMPORAL_FEATURE_FILE = PROJECT_ROOT / 'data/processed/all_traffic_features_temporal.csv'
df.to_csv(TEMPORAL_FEATURE_FILE, index=False)

print('Rows:', len(df))
print('Videos:', df['video_id'].nunique())
print('Saved:', TEMPORAL_FEATURE_FILE)


## 7. Prepare the machine-learning data

The model receives only traffic and temporal features. Video id, timestamps, split names, and labels are intentionally excluded from the inputs to avoid leakage.


In [ ]:
FEATURE_COLUMNS = [
    # Current traffic state
    'unique_vehicle_count',
    'mean_vehicle_count_per_frame',
    'mean_normalized_speed',
    'median_normalized_speed',
    'stopped_vehicle_count',
    'stopped_ratio',
    'bbox_coverage_proxy',
    'max_bbox_coverage_proxy',
    'mean_detection_confidence',
    'car_unique',
    'motorcycle_unique',
    'bus_unique',
    'truck_unique',

    # Change from the previous window
    'delta_unique_vehicle_count',
    'delta_mean_vehicle_count_per_frame',
    'delta_mean_normalized_speed',
    'delta_median_normalized_speed',
    'delta_stopped_vehicle_count',
    'delta_stopped_ratio',
    'delta_bbox_coverage_proxy',

    # Short recent history
    'roll3_unique_vehicle_count',
    'roll3_mean_vehicle_count_per_frame',
    'roll3_mean_normalized_speed',
    'roll3_median_normalized_speed',
    'roll3_stopped_vehicle_count',
    'roll3_stopped_ratio',
    'roll3_bbox_coverage_proxy',
]

train_df = df[df['split'] == 'train'].copy()
val_df = df[df['split'] == 'val'].copy()
test_df = df[df['split'] == 'test'].copy()

X_train, y_train = train_df[FEATURE_COLUMNS], train_df['label'].astype(int)
X_val, y_val = val_df[FEATURE_COLUMNS], val_df['label'].astype(int)
X_test, y_test = test_df[FEATURE_COLUMNS], test_df['label'].astype(int)

negative = int((y_train == 0).sum())
positive = int((y_train == 1).sum())
imbalance_ratio = negative / positive

print('Feature count:', len(FEATURE_COLUMNS))
print('Train:', X_train.shape, 'Validation:', X_val.shape, 'Test:', X_test.shape)
print('Train normal / incident:', negative, '/', positive)
print('Class imbalance ratio:', round(imbalance_ratio, 3))


## 8. Train Random Forest and XGBoost baselines

The incident class is much smaller than the normal class, so both models use class weighting. F2 is used when choosing thresholds because missing incidents is more costly for this prototype than generating an occasional false alarm.


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    average_precision_score,
    classification_report,
    confusion_matrix,
    f1_score,
    fbeta_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.pipeline import Pipeline
from xgboost import XGBClassifier

rf_model = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('classifier', RandomForestClassifier(
        n_estimators=500,
        max_depth=16,
        min_samples_leaf=2,
        class_weight='balanced_subsample',
        n_jobs=-1,
        random_state=42,
    )),
])

xgb_model = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('classifier', XGBClassifier(
        n_estimators=600,
        max_depth=6,
        learning_rate=0.04,
        subsample=0.85,
        colsample_bytree=0.85,
        scale_pos_weight=imbalance_ratio,
        eval_metric='logloss',
        n_jobs=-1,
        random_state=42,
    )),
])

rf_model.fit(X_train, y_train)
xgb_model.fit(X_train, y_train)
print('Random Forest and XGBoost trained.')


In [ ]:
def best_threshold_from_probabilities(y_true, probabilities, start=0.05, stop=0.96):
    """Choose the threshold that maximizes F2."""
    best_threshold, best_f2 = 0.5, -1.0
    for threshold in np.arange(start, stop, 0.01):
        prediction = (probabilities >= threshold).astype(int)
        score = fbeta_score(y_true, prediction, beta=2, zero_division=0)
        if score > best_f2:
            best_threshold = float(threshold)
            best_f2 = float(score)
    return best_threshold, best_f2


def print_metrics(name, y_true, probabilities, threshold):
    prediction = (probabilities >= threshold).astype(int)
    print('\n' + '=' * 70)
    print(name)
    print('=' * 70)
    print('Precision:', precision_score(y_true, prediction, zero_division=0))
    print('Recall:', recall_score(y_true, prediction, zero_division=0))
    print('F1:', f1_score(y_true, prediction, zero_division=0))
    print('F2:', fbeta_score(y_true, prediction, beta=2, zero_division=0))
    print('Average Precision:', average_precision_score(y_true, probabilities))
    print('ROC AUC:', roc_auc_score(y_true, probabilities))
    print('\nConfusion matrix:')
    print(confusion_matrix(y_true, prediction))
    return prediction

rf_val_prob = rf_model.predict_proba(X_val)[:, 1]
xgb_val_prob = xgb_model.predict_proba(X_val)[:, 1]
rf_val_threshold, rf_val_f2 = best_threshold_from_probabilities(y_val, rf_val_prob)
xgb_val_threshold, xgb_val_f2 = best_threshold_from_probabilities(y_val, xgb_val_prob)

print('RF validation threshold / F2:', rf_val_threshold, rf_val_f2)
print('XGB validation threshold / F2:', xgb_val_threshold, xgb_val_f2)


### Why grouped cross-validation is used for the final choice

The simple validation split contains only a small number of anomalous videos, so its score can be overly optimistic for a particular camera scene. The final model selection therefore uses **5-fold grouped cross-validation by `video_id`**. A whole video stays in one fold, which prevents adjacent windows from leaking between training and validation.


In [ ]:
from sklearn.base import clone
from sklearn.model_selection import StratifiedGroupKFold

# Development data uses train + validation. The test split remains untouched.
dev_df = df[df['split'].isin(['train', 'val'])].copy()
X_dev = dev_df[FEATURE_COLUMNS]
y_dev = dev_df['label'].astype(int)
groups = dev_df['video_id'].astype(int)

cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)


def grouped_oof_predictions(base_model, X, y, groups, name):
    """Generate one out-of-fold probability for every development row."""
    oof_probability = np.zeros(len(X), dtype=float)
    print('\n' + '=' * 70)
    print(name)
    print('=' * 70)

    for fold, (train_idx, valid_idx) in enumerate(cv.split(X, y, groups), start=1):
        model = clone(base_model)
        model.fit(X.iloc[train_idx], y.iloc[train_idx])
        oof_probability[valid_idx] = model.predict_proba(X.iloc[valid_idx])[:, 1]
        print(
            f'Fold {fold}: train={len(train_idx)}, valid={len(valid_idx)}, '
            f'valid incidents={int(y.iloc[valid_idx].sum())}'
        )

    return oof_probability

rf_oof = grouped_oof_predictions(rf_model, X_dev, y_dev, groups, 'Random Forest grouped CV')
xgb_oof = grouped_oof_predictions(xgb_model, X_dev, y_dev, groups, 'XGBoost grouped CV')

rf_cv_threshold, rf_cv_f2 = best_threshold_from_probabilities(y_dev, rf_oof)
xgb_cv_threshold, xgb_cv_f2 = best_threshold_from_probabilities(y_dev, xgb_oof)

print_metrics('RANDOM FOREST - GROUPED CV', y_dev, rf_oof, rf_cv_threshold)
print_metrics('XGBOOST - GROUPED CV', y_dev, xgb_oof, xgb_cv_threshold)
print('\nRF threshold / F2:', rf_cv_threshold, rf_cv_f2)
print('XGB threshold / F2:', xgb_cv_threshold, xgb_cv_f2)


In [ ]:
if xgb_cv_f2 > rf_cv_f2:
    selected_model = xgb_model
    selected_model_name = 'XGBoost'
    selected_oof = xgb_oof
else:
    selected_model = rf_model
    selected_model_name = 'Random Forest'
    selected_oof = rf_oof

print('Selected model:', selected_model_name)


## 9. Temporal confirmation

A single noisy 5-second prediction should not immediately trigger an alert. The following step compares a few simple temporal policies (raw prediction, two consecutive positives, two of the last three, or the mean probability of the last three windows). The policy and threshold are chosen **only from out-of-fold development predictions**.


In [ ]:
def temporal_predictions(frame, probabilities, threshold, rule):
    """Apply a temporal confirmation rule independently inside each video."""
    temp = frame[['video_id', 'window_start_sec', 'label']].copy().reset_index(drop=True)
    temp['probability'] = probabilities
    predictions = np.zeros(len(temp), dtype=int)

    for _, group in temp.groupby('video_id', sort=False):
        indices = group.index.to_list()
        probs = group['probability'].to_numpy()
        raw = (probs >= threshold).astype(int)

        for local_i, global_i in enumerate(indices):
            if rule == 'raw':
                decision = raw[local_i]
            elif rule == 'two_consecutive':
                decision = int(local_i >= 1 and raw[local_i] == 1 and raw[local_i - 1] == 1)
            elif rule == 'two_of_three':
                recent = raw[max(0, local_i - 2): local_i + 1]
                decision = int(recent.sum() >= 2)
            elif rule == 'mean3':
                recent_probs = probs[max(0, local_i - 2): local_i + 1]
                decision = int(recent_probs.mean() >= threshold)
            else:
                raise ValueError(f'Unknown rule: {rule}')

            predictions[global_i] = decision

    return predictions


rules = ['raw', 'two_consecutive', 'two_of_three', 'mean3']
best_policy = None

for rule in rules:
    for threshold in np.arange(0.05, 0.81, 0.01):
        pred = temporal_predictions(dev_df, selected_oof, threshold, rule)
        score = fbeta_score(y_dev, pred, beta=2, zero_division=0)

        if best_policy is None or score > best_policy['f2']:
            best_policy = {
                'rule': rule,
                'threshold': float(threshold),
                'f2': float(score),
                'precision': float(precision_score(y_dev, pred, zero_division=0)),
                'recall': float(recall_score(y_dev, pred, zero_division=0)),
                'f1': float(f1_score(y_dev, pred, zero_division=0)),
            }

print('Best temporal policy:', best_policy)


## 10. Final training and held-out test

The selected model is retrained on all development videos (`train + validation`). The held-out test videos are used only once here for the final evaluation.


In [ ]:
FINAL_MODEL_NAME = selected_model_name
FINAL_THRESHOLD = best_policy['threshold']
FINAL_TEMPORAL_RULE = best_policy['rule']
FINAL_MODEL = clone(selected_model)
FINAL_MODEL.fit(X_dev, y_dev)

final_test_probability = FINAL_MODEL.predict_proba(X_test)[:, 1]
final_test_prediction = temporal_predictions(
    test_df,
    final_test_probability,
    FINAL_THRESHOLD,
    FINAL_TEMPORAL_RULE,
)

print(classification_report(y_test, final_test_prediction, digits=4, zero_division=0))
print('Confusion Matrix:')
print(confusion_matrix(y_test, final_test_prediction))
print('Precision:', precision_score(y_test, final_test_prediction, zero_division=0))
print('Recall:', recall_score(y_test, final_test_prediction, zero_division=0))
print('F1:', f1_score(y_test, final_test_prediction, zero_division=0))
print('F2:', fbeta_score(y_test, final_test_prediction, beta=2, zero_division=0))
print('Average Precision:', average_precision_score(y_test, final_test_probability))
print('ROC AUC:', roc_auc_score(y_test, final_test_probability))


### Results from the current run

The current run selected **Random Forest**. On the held-out test videos, after temporal confirmation, the measured performance was:

| Metric | Value |
|---|---:|
| Precision | 0.2430 |
| Recall | 0.5411 |
| F1 | 0.3353 |
| F2 | 0.4344 |
| Average Precision | 0.2288 |
| ROC-AUC | 0.7574 |

Confusion matrix: `[[2124, 349], [95, 112]]`.

These numbers are kept here as a record of the experiment. If the extraction settings, model hyperparameters, or split files change, the notebook should be rerun and this table updated.


## 11. Save the model and test predictions

The saved artifact includes the classifier, feature list, decision threshold, and temporal rule so inference uses exactly the same settings as evaluation.


In [ ]:
MODEL_DIR = PROJECT_ROOT / 'models'
MODEL_DIR.mkdir(parents=True, exist_ok=True)
MODEL_FILE = MODEL_DIR / 'incident_detector.joblib'

final_artifact = {
    'model': FINAL_MODEL,
    'model_name': FINAL_MODEL_NAME,
    'feature_columns': FEATURE_COLUMNS,
    'threshold': FINAL_THRESHOLD,
    'temporal_rule': FINAL_TEMPORAL_RULE,
    'window_seconds': WINDOW_SECONDS,
    'cv_f2': best_policy['f2'],
    'cv_precision': best_policy['precision'],
    'cv_recall': best_policy['recall'],
}
joblib.dump(final_artifact, MODEL_FILE)

final_test_results = test_df[
    ['video_id', 'window_start_sec', 'window_end_sec', 'label']
].copy()
final_test_results['incident_probability'] = final_test_probability
final_test_results['prediction'] = final_test_prediction

TEST_RESULTS_FILE = PROJECT_ROOT / 'outputs/final_incident_test_results.csv'
TEST_RESULTS_FILE.parent.mkdir(parents=True, exist_ok=True)
final_test_results.to_csv(TEST_RESULTS_FILE, index=False)

print('Model saved to:', MODEL_FILE)
print('Test predictions saved to:', TEST_RESULTS_FILE)


resume from the other lab

In [ ]:
from pathlib import Path
import json
import joblib
import numpy as np
import pandas as pd


# Project files already created during training
PROJECT_ROOT = Path(
    "/content/drive/MyDrive/5piproject"
)

MODEL_FILE = (
    PROJECT_ROOT
    / "models"
    / "incident_detector.joblib"
)

TEMPORAL_FEATURE_FILE = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "all_traffic_features_temporal.csv"
)


print("Model exists:", MODEL_FILE.exists())
print("Feature data exists:", TEMPORAL_FEATURE_FILE.exists())


if not MODEL_FILE.exists():
    raise FileNotFoundError(
        f"Saved model not found: {MODEL_FILE}"
    )

if not TEMPORAL_FEATURE_FILE.exists():
    raise FileNotFoundError(
        f"Temporal feature file not found: {TEMPORAL_FEATURE_FILE}"
    )


# Load the model artifact saved during training
artifact = joblib.load(MODEL_FILE)

incident_model = artifact["model"]
incident_features = artifact["feature_columns"]
incident_threshold = float(
    artifact["threshold"]
)

# Compatible with older saved versions too
temporal_rule = artifact.get(
    "temporal_rule",
    "raw"
)

WINDOW_SECONDS = float(
    artifact.get(
        "window_seconds",
        5.0
    )
)


# Load the already-extracted traffic features
MASTER_DATA = pd.read_csv(
    TEMPORAL_FEATURE_FILE
)


print("\n✅ Saved model loaded successfully")
print("Model:", artifact.get("model_name", "Unknown"))
print("Threshold:", incident_threshold)
print("Temporal rule:", temporal_rule)
print("Number of features:", len(incident_features))
print("Traffic rows:", len(MASTER_DATA))

**bold text**## 12. Inference helper and LangChain tool

The LangChain layer does **not** detect the incident itself. The trained classifier makes the prediction; the agent reads the model result and decides how to communicate or act on it.


In [ ]:
artifact = joblib.load(MODEL_FILE)
incident_model = artifact['model']
incident_features = artifact['feature_columns']
incident_threshold = float(artifact['threshold'])
temporal_rule = artifact['temporal_rule']
MASTER_DATA = pd.read_csv(TEMPORAL_FEATURE_FILE)


def confirm_incident(probabilities, threshold, rule):
    probabilities = np.asarray(probabilities, dtype=float)
    raw = (probabilities >= threshold).astype(int)

    if rule == 'raw':
        return bool(raw[-1])
    if rule == 'two_consecutive':
        return len(raw) >= 2 and bool(raw[-1] and raw[-2])
    if rule == 'two_of_three':
        return bool(raw[-3:].sum() >= 2)
    if rule == 'mean3':
        return bool(probabilities[-3:].mean() >= threshold)
    raise ValueError(f'Unknown rule: {rule}')

print('Loaded model:', artifact['model_name'])
print('Threshold:', incident_threshold)
print('Temporal rule:', temporal_rule)


In [ ]:
from langchain.tools import tool


@tool
def detect_traffic_incident(video_id: int, window_start_sec: float) -> str:
    """Predict whether a traffic incident is confirmed near a requested 5-second window."""
    video_data = (
        MASTER_DATA[MASTER_DATA['video_id'] == int(video_id)]
        .sort_values('window_start_sec')
        .copy()
    )
    if video_data.empty:
        return json.dumps({'error': f'Unknown video {video_id}'})

    nearest_index = (
        video_data['window_start_sec']
        .sub(float(window_start_sec))
        .abs()
        .idxmin()
    )
    current_time = float(MASTER_DATA.loc[nearest_index, 'window_start_sec'])

    # Keep only causal history: current window + at most two previous windows.
    recent = video_data[video_data['window_start_sec'] <= current_time].tail(3).copy()
    X_recent = recent[incident_features]
    probabilities = incident_model.predict_proba(X_recent)[:, 1]
    raw_predictions = (probabilities >= incident_threshold).astype(int)
    confirmed = confirm_incident(probabilities, incident_threshold, temporal_rule)
    current_row = recent.iloc[-1]

    result = {
        'video_id': int(video_id),
        'window_start_sec': float(current_row['window_start_sec']),
        'window_end_sec': float(current_row['window_end_sec']),
        'model': artifact['model_name'],
        'threshold': round(incident_threshold, 4),
        'temporal_rule': temporal_rule,
        'current_probability': round(float(probabilities[-1]), 4),
        'recent_probabilities': [round(float(p), 4) for p in probabilities],
        'recent_raw_predictions': [
            'INCIDENT' if int(p) else 'NORMAL' for p in raw_predictions
        ],
        'detector_verdict': 'INCIDENT' if confirmed else 'NORMAL',
        'traffic_evidence': {
            'unique_vehicle_count': int(current_row['unique_vehicle_count']),
            'mean_vehicle_count_per_frame': round(float(current_row['mean_vehicle_count_per_frame']), 3),
            'median_normalized_speed': round(float(current_row['median_normalized_speed']), 3),
            'stopped_vehicle_count': int(current_row['stopped_vehicle_count']),
            'stopped_ratio': round(float(current_row['stopped_ratio']), 3),
            'bbox_coverage_proxy': round(float(current_row['bbox_coverage_proxy']), 4),
        },
    }
    return json.dumps(result, indent=2)


## 13. Connect the tool to the language-model agent

The API key is read from **Colab Secrets** under `APINEX_API_KEY`; it is never written directly into the notebook. APInex is used here because it exposes an OpenAI-compatible endpoint.


In [ ]:
from google.colab import userdata
from langchain.agents import create_agent
from langchain_openai import ChatOpenAI

APINEX_API_KEY = userdata.get('APINEX_API_KEY')
if not APINEX_API_KEY:
    raise ValueError('APINEX_API_KEY is missing from Colab Secrets.')

llm = ChatOpenAI(
    model='free/gpt-6-luna',
    api_key=APINEX_API_KEY,
    base_url='https://api.apinex.bond/v1',
    temperature=0,
)


In [ ]:
SYSTEM_PROMPT = """
You are the Incident Detection Agent in an intelligent multi-agent traffic-management system.

Always call detect_traffic_incident before making a decision.
The tool result comes from a trained traffic incident classifier with temporal confirmation.

Rules:
- detector_verdict is the final model decision.
- current_probability is the classifier's current incident probability.
- YOLO detects vehicles; it does not directly detect accidents.
- ByteTrack provides vehicle trajectories.
- unique_vehicle_count means unique tracked ids during the window, not necessarily simultaneous vehicles.
- normalized speed is an image-motion proxy, not km/h.
- stopped_vehicle_count is a supporting heuristic, not proof of a crash.
- Do not invent an incident type. The model currently predicts only NORMAL or INCIDENT.
- There is no trained severity model, so severity must be UNKNOWN.
- If an incident is confirmed, recommend notifying the Traffic Signal Agent and continuing monitoring.
- If the result is normal, recommend normal monitoring.

Return exactly:
STATUS:
SEVERITY:
INCIDENT_PROBABILITY:
TEMPORAL_EVIDENCE:
TRAFFIC_EVIDENCE:
RECOMMENDED_ACTION:
"""

agent = create_agent(
    model=llm,
    tools=[detect_traffic_incident],
    system_prompt=SYSTEM_PROMPT,
)

print('Incident Detection Agent ready.')


## 14. Final demo on held-out test examples

For the demo we choose one correctly detected incident and one correctly detected normal window. This is only for presentation; the full held-out metrics above remain the real evaluation of the model.


In [ ]:
from pathlib import Path

import joblib
import numpy as np
import pandas as pd


# ============================================================
# Paths
# ============================================================

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/5piproject"
)

MODEL_FILE = (
    PROJECT_ROOT
    / "models"
    / "incident_detector.joblib"
)

TEMPORAL_FEATURE_FILE = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "all_traffic_features_temporal.csv"
)

FINAL_TEST_FILE = (
    PROJECT_ROOT
    / "outputs"
    / "final_incident_test_results.csv"
)


print("Model exists:", MODEL_FILE.exists())
print(
    "Temporal dataset exists:",
    TEMPORAL_FEATURE_FILE.exists()
)


if not MODEL_FILE.exists():
    raise FileNotFoundError(
        f"Model not found: {MODEL_FILE}"
    )

if not TEMPORAL_FEATURE_FILE.exists():
    raise FileNotFoundError(
        f"Feature file not found: {TEMPORAL_FEATURE_FILE}"
    )


# ============================================================
# Load trained detector
# ============================================================

artifact = joblib.load(
    MODEL_FILE
)

print("\nSaved artifact keys:")
print(artifact.keys())


incident_model = artifact["model"]

incident_features = artifact[
    "feature_columns"
]

incident_threshold = float(
    artifact["threshold"]
)


# Important:
# We want the SAME temporal policy used in your final evaluation.
temporal_rule = artifact.get(
    "temporal_rule"
)


if temporal_rule is None:
    raise ValueError(
        "The saved model does not contain 'temporal_rule'. "
        "Do not guess it, because that would change your "
        "final predictions. Check the old notebook for the "
        "value of FINAL_TEMPORAL_RULE."
    )


print("\n✅ Model loaded")
print(
    "Model:",
    artifact.get(
        "model_name",
        "Unknown"
    )
)
print(
    "Threshold:",
    incident_threshold
)
print(
    "Temporal rule:",
    temporal_rule
)
print(
    "Features:",
    len(incident_features)
)


# ============================================================
# Load extracted temporal features
# ============================================================

MASTER_DATA = pd.read_csv(
    TEMPORAL_FEATURE_FILE
)


MASTER_DATA = (
    MASTER_DATA
    .sort_values(
        [
            "video_id",
            "window_start_sec"
        ]
    )
    .reset_index(drop=True)
)


print(
    "\nTotal traffic windows:",
    len(MASTER_DATA)
)


# ============================================================
# Recover the original held-out test set
# ============================================================

test_df = (
    MASTER_DATA[
        MASTER_DATA["split"] == "test"
    ]
    .copy()
    .sort_values(
        [
            "video_id",
            "window_start_sec"
        ]
    )
    .reset_index(drop=True)
)


print(
    "Test windows:",
    len(test_df)
)

print(
    "Real incident windows:",
    int(test_df["label"].sum())
)


# ============================================================
# Run the saved Random Forest
# ============================================================

X_test = test_df[
    incident_features
].copy()


test_probabilities = (
    incident_model
    .predict_proba(
        X_test
    )[:, 1]
)


test_df[
    "incident_probability"
] = test_probabilities


# ============================================================
# Apply temporal confirmation
# separately for every video
# ============================================================

final_predictions = np.zeros(
    len(test_df),
    dtype=int
)


for _, group in test_df.groupby(
    "video_id",
    sort=False
):

    group = group.sort_values(
        "window_start_sec"
    )

    indices = group.index.to_numpy()

    probabilities = group[
        "incident_probability"
    ].to_numpy()


    raw = (
        probabilities
        >= incident_threshold
    ).astype(int)


    video_predictions = []


    for i in range(
        len(group)
    ):

        # ------------------------------------
        # No temporal confirmation
        # ------------------------------------

        if temporal_rule == "raw":

            decision = int(
                raw[i]
            )


        # ------------------------------------
        # Current + previous must be positive
        # ------------------------------------

        elif temporal_rule == "two_consecutive":

            decision = int(
                i >= 1
                and
                raw[i] == 1
                and
                raw[i - 1] == 1
            )


        # ------------------------------------
        # At least 2 positives in last 3
        # ------------------------------------

        elif temporal_rule == "two_of_three":

            recent = raw[
                max(
                    0,
                    i - 2
                ):
                i + 1
            ]

            decision = int(
                recent.sum() >= 2
            )


        # ------------------------------------
        # Mean probability of last 3
        # ------------------------------------

        elif temporal_rule == "mean3":

            recent_probabilities = (
                probabilities[
                    max(
                        0,
                        i - 2
                    ):
                    i + 1
                ]
            )

            decision = int(
                recent_probabilities.mean()
                >= incident_threshold
            )


        else:

            raise ValueError(
                "Unknown temporal rule: "
                f"{temporal_rule}"
            )


        video_predictions.append(
            decision
        )


    final_predictions[
        indices
    ] = video_predictions


test_df[
    "prediction"
] = final_predictions


# ============================================================
# Recreate final_test_results
# ============================================================

final_test_results = test_df[
    [
        "video_id",
        "window_start_sec",
        "window_end_sec",
        "label",
        "incident_probability",
        "prediction"
    ]
].copy()


# ============================================================
# Save it permanently this time
# ============================================================

FINAL_TEST_FILE.parent.mkdir(
    parents=True,
    exist_ok=True
)


final_test_results.to_csv(
    FINAL_TEST_FILE,
    index=False
)


print(
    "\n✅ final_test_results rebuilt"
)

print(
    "Saved to:",
    FINAL_TEST_FILE
)


# ============================================================
# Quick verification
# ============================================================

print("\nPrediction counts:")

print(
    pd.crosstab(
        final_test_results["label"],
        final_test_results["prediction"],
        rownames=["Actual"],
        colnames=["Predicted"]
    )
)

In [ ]:
true_positives = final_test_results[
    (final_test_results['label'] == 1) & (final_test_results['prediction'] == 1)
].sort_values('incident_probability', ascending=False)

true_negatives = final_test_results[
    (final_test_results['label'] == 0) & (final_test_results['prediction'] == 0)
].sort_values('incident_probability')

if true_positives.empty or true_negatives.empty:
    raise RuntimeError('Could not find both a true-positive and a true-negative demo sample.')

demo_incident = true_positives.iloc[0]
demo_normal = true_negatives.iloc[0]

DEMO_INCIDENT_VIDEO = int(demo_incident['video_id'])
DEMO_INCIDENT_TIME = float(demo_incident['window_start_sec'])
DEMO_NORMAL_VIDEO = int(demo_normal['video_id'])
DEMO_NORMAL_TIME = float(demo_normal['window_start_sec'])

print('Incident demo:', DEMO_INCIDENT_VIDEO, DEMO_INCIDENT_TIME,
      'p=', round(float(demo_incident['incident_probability']), 4))
print('Normal demo:', DEMO_NORMAL_VIDEO, DEMO_NORMAL_TIME,
      'p=', round(float(demo_normal['incident_probability']), 4))


In [ ]:
incident_response = agent.invoke({
    'messages': [{
        'role': 'user',
        'content': (
            f'Analyze traffic video {DEMO_INCIDENT_VIDEO} at {DEMO_INCIDENT_TIME} seconds. '
            'Use the trained detector and decide whether an incident is confirmed.'
        ),
    }]
})
print(incident_response['messages'][-1].content)


In [ ]:
normal_response = agent.invoke({
    'messages': [{
        'role': 'user',
        'content': (
            f'Analyze traffic video {DEMO_NORMAL_VIDEO} at {DEMO_NORMAL_TIME} seconds. '
            'Use the trained detector and decide whether an incident is confirmed.'
        ),
    }]
})
print(normal_response['messages'][-1].content)


testtttttttttttt


In [ ]:
# ============================================================
# EXTERNAL MP4 TEST — COMPLETE PIPELINE IN ONE CELL
# ============================================================

from pathlib import Path
import json
import joblib
import cv2
import numpy as np
import pandas as pd
import torch

from google.colab import files, userdata
from ultralytics import YOLO
from langchain.tools import tool
from langchain.agents import create_agent
from langchain_openai import ChatOpenAI


# ============================================================
# 1. LOAD SAVED INCIDENT DETECTOR
# ============================================================

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/5piproject"
)

MODEL_FILE = (
    PROJECT_ROOT
    / "models"
    / "incident_detector.joblib"
)


if not MODEL_FILE.exists():
    raise FileNotFoundError(
        f"Saved detector not found: {MODEL_FILE}"
    )


artifact = joblib.load(
    MODEL_FILE
)

incident_model = artifact["model"]
incident_features = artifact["feature_columns"]
incident_threshold = float(
    artifact["threshold"]
)

temporal_rule = artifact[
    "temporal_rule"
]


print("✅ Saved detector loaded")
print("Model:", artifact.get("model_name"))
print("Threshold:", incident_threshold)
print("Temporal rule:", temporal_rule)


# ============================================================
# 2. SETTINGS — SAME AS TRAINING
# ============================================================

WINDOW_SECONDS = 5.0
TARGET_FPS = 5.0

CONFIDENCE_THRESHOLD = 0.25

STOPPED_SPEED_THRESHOLD = 0.08
MIN_STOPPED_OBSERVATIONS = 5


CAR = 2
MOTORCYCLE = 3
BUS = 5
TRUCK = 7

VEHICLE_CLASSES = [
    CAR,
    MOTORCYCLE,
    BUS,
    TRUCK
]


DEVICE = (
    0
    if torch.cuda.is_available()
    else "cpu"
)


print(
    "Device:",
    DEVICE
)


# ============================================================
# 3. UPLOAD MP4
# ============================================================

print(
    "\n📤 Upload your MP4 traffic video..."
)

uploaded = files.upload()


if not uploaded:
    raise RuntimeError(
        "No video uploaded."
    )


filename, file_bytes = next(
    iter(
        uploaded.items()
    )
)


if not filename.lower().endswith(
    ".mp4"
):
    raise ValueError(
        "Please upload an MP4 video."
    )


UPLOAD_DIR = Path(
    "/content/external_incident_test"
)

UPLOAD_DIR.mkdir(
    parents=True,
    exist_ok=True
)


VIDEO_PATH = (
    UPLOAD_DIR
    / Path(filename).name
)


VIDEO_PATH.write_bytes(
    file_bytes
)


print(
    "✅ Uploaded:",
    VIDEO_PATH
)


# ============================================================
# 4. VIDEO INFORMATION
# ============================================================

cap = cv2.VideoCapture(
    str(VIDEO_PATH)
)


if not cap.isOpened():
    raise RuntimeError(
        "OpenCV could not open the video."
    )


original_fps = cap.get(
    cv2.CAP_PROP_FPS
)

frame_count = cap.get(
    cv2.CAP_PROP_FRAME_COUNT
)

width = int(
    cap.get(
        cv2.CAP_PROP_FRAME_WIDTH
    )
)

height = int(
    cap.get(
        cv2.CAP_PROP_FRAME_HEIGHT
    )
)

cap.release()


if original_fps <= 0:
    raise RuntimeError(
        "Invalid FPS."
    )


duration = (
    frame_count
    / original_fps
)


vid_stride = max(
    1,
    round(
        original_fps
        / TARGET_FPS
    )
)


processed_fps = (
    original_fps
    / vid_stride
)


frame_area = (
    width
    * height
)


print(
    "\n🎥 VIDEO INFORMATION"
)

print(
    "Resolution:",
    f"{width}x{height}"
)

print(
    "Original FPS:",
    round(
        original_fps,
        2
    )
)

print(
    "Duration:",
    round(
        duration,
        2
    ),
    "seconds"
)

print(
    "Processing FPS:",
    round(
        processed_fps,
        2
    )
)


# ============================================================
# 5. YOLO + BYTETRACK
# ============================================================

print(
    "\n🚗 Running YOLO + ByteTrack..."
)


yolo_model = YOLO(
    "yolo11n.pt"
)


frame_rows = []
detection_rows = []


results = yolo_model.track(

    source=str(
        VIDEO_PATH
    ),

    tracker=
        "bytetrack.yaml",

    persist=True,

    stream=True,

    vid_stride=
        vid_stride,

    conf=
        CONFIDENCE_THRESHOLD,

    classes=
        VEHICLE_CLASSES,

    device=
        DEVICE,

    verbose=False,

    save=False
)


for processed_frame, result in enumerate(
    results
):

    source_time = (
        processed_frame
        / processed_fps
    )


    boxes = result.boxes

    vehicle_count = 0
    bbox_area_sum = 0.0

    confidences = []


    if boxes is not None:

        for box in boxes:

            if box.id is None:
                continue


            track_id = int(
                box.id[0].item()
            )


            class_id = int(
                box.cls[0].item()
            )


            confidence = float(
                box.conf[0].item()
            )


            x1, y1, x2, y2 = (
                box.xyxy[0]
                .cpu()
                .numpy()
                .tolist()
            )


            bbox_width = (
                x2 - x1
            )

            bbox_height = (
                y2 - y1
            )


            bbox_area = (
                bbox_width
                * bbox_height
            )


            center_x = (
                x1 + x2
            ) / 2

            center_y = (
                y1 + y2
            ) / 2


            vehicle_count += 1

            bbox_area_sum += (
                bbox_area
            )

            confidences.append(
                confidence
            )


            detection_rows.append({

                "processed_frame":
                    processed_frame,

                "source_time_sec":
                    source_time,

                "track_id":
                    track_id,

                "class_id":
                    class_id,

                "confidence":
                    confidence,

                "center_x":
                    center_x,

                "center_y":
                    center_y,

                "bbox_height":
                    bbox_height,

                "bbox_area":
                    bbox_area
            })


    frame_rows.append({

        "processed_frame":
            processed_frame,

        "source_time_sec":
            source_time,

        "vehicle_count":
            vehicle_count,

        "bbox_coverage":
            min(
                bbox_area_sum
                / frame_area,
                1.0
            ),

        "mean_detection_confidence":
            (
                float(
                    np.mean(
                        confidences
                    )
                )
                if confidences
                else 0.0
            )
    })


print(
    "✅ YOLO + ByteTrack finished"
)


# ============================================================
# 6. CREATE DATAFRAMES
# ============================================================

frames = pd.DataFrame(
    frame_rows
)

detections = pd.DataFrame(
    detection_rows
)


if frames.empty:
    raise RuntimeError(
        "No frames processed."
    )


print(
    "Processed frames:",
    len(frames)
)

print(
    "Tracked detections:",
    len(detections)
)


# ============================================================
# 7. CREATE 5-SECOND WINDOWS
# ============================================================

frames[
    "window_start_sec"
] = (

    np.floor(

        frames[
            "source_time_sec"
        ]

        / WINDOW_SECONDS

    )

    * WINDOW_SECONDS

).round(3)


if not detections.empty:

    detections[
        "window_start_sec"
    ] = (

        np.floor(

            detections[
                "source_time_sec"
            ]

            / WINDOW_SECONDS

        )

        * WINDOW_SECONDS

    ).round(3)


# ============================================================
# 8. VEHICLE MOTION
# ============================================================

if not detections.empty:

    detections = (

        detections
        .sort_values(
            [
                "track_id",
                "source_time_sec"
            ]
        )
        .reset_index(
            drop=True
        )
    )


    detections[
        "prev_x"
    ] = (

        detections
        .groupby(
            "track_id"
        )[
            "center_x"
        ]
        .shift(1)
    )


    detections[
        "prev_y"
    ] = (

        detections
        .groupby(
            "track_id"
        )[
            "center_y"
        ]
        .shift(1)
    )


    detections[
        "prev_time"
    ] = (

        detections
        .groupby(
            "track_id"
        )[
            "source_time_sec"
        ]
        .shift(1)
    )


    detections[
        "distance_px"
    ] = np.sqrt(

        (
            detections[
                "center_x"
            ]
            -
            detections[
                "prev_x"
            ]
        ) ** 2

        +

        (
            detections[
                "center_y"
            ]
            -
            detections[
                "prev_y"
            ]
        ) ** 2
    )


    detections[
        "dt"
    ] = (

        detections[
            "source_time_sec"
        ]

        -

        detections[
            "prev_time"
        ]
    )


    detections.loc[
        detections["dt"] <= 0,
        "dt"
    ] = np.nan


    detections[
        "speed_px_sec"
    ] = (

        detections[
            "distance_px"
        ]

        /

        detections[
            "dt"
        ]
    )


    # Relative image motion.
    # This is NOT km/h.
    detections[
        "normalized_speed"
    ] = (

        detections[
            "speed_px_sec"
        ]

        /

        detections[
            "bbox_height"
        ].clip(
            lower=1
        )
    )


    detections.replace(
        [
            np.inf,
            -np.inf
        ],
        np.nan,
        inplace=True
    )


    track_window = (

        detections
        .groupby(
            [
                "window_start_sec",
                "track_id"
            ],
            as_index=False
        )
        .agg(

            observations=(
                "track_id",
                "size"
            ),

            median_normalized_speed=(
                "normalized_speed",
                "median"
            )
        )
    )


    track_window[
        "potentially_stopped"
    ] = (

        (
            track_window[
                "observations"
            ]
            >=
            MIN_STOPPED_OBSERVATIONS
        )

        &

        (
            track_window[
                "median_normalized_speed"
            ]
            <
            STOPPED_SPEED_THRESHOLD
        )
    )


else:

    track_window = pd.DataFrame(

        columns=[

            "window_start_sec",

            "track_id",

            "observations",

            "median_normalized_speed",

            "potentially_stopped"
        ]
    )


# ============================================================
# 9. EXTRACT FEATURES PER 5-SECOND WINDOW
# ============================================================

feature_rows = []


for start in np.arange(
    0,
    duration,
    WINDOW_SECONDS
):

    start = round(
        float(start),
        3
    )


    end = min(
        start
        + WINDOW_SECONDS,
        duration
    )


    frame_group = (

        frames[

            frames[
                "window_start_sec"
            ]
            ==
            start

        ]
    )


    if detections.empty:

        detection_group = (
            detections
        )

    else:

        detection_group = (

            detections[

                detections[
                    "window_start_sec"
                ]
                ==
                start

            ]
        )


    if frame_group.empty:

        mean_vehicle_count = 0.0
        coverage = 0.0
        max_coverage = 0.0
        mean_confidence = 0.0

    else:

        mean_vehicle_count = float(

            frame_group[
                "vehicle_count"
            ].mean()

        )


        coverage = float(

            frame_group[
                "bbox_coverage"
            ].mean()

        )


        max_coverage = float(

            frame_group[
                "bbox_coverage"
            ].max()

        )


        mean_confidence = float(

            frame_group[
                "mean_detection_confidence"
            ].mean()

        )


    if detection_group.empty:

        unique_vehicles = 0

        mean_speed = 0.0

        median_speed = 0.0


        car_unique = 0
        motorcycle_unique = 0
        bus_unique = 0
        truck_unique = 0


    else:

        unique_vehicles = int(

            detection_group[
                "track_id"
            ].nunique()

        )


        speeds = (

            detection_group[
                "normalized_speed"
            ]
            .dropna()

        )


        mean_speed = (

            float(
                speeds.mean()
            )

            if len(
                speeds
            )

            else 0.0

        )


        median_speed = (

            float(
                speeds.median()
            )

            if len(
                speeds
            )

            else 0.0

        )


        def count_class(
            class_id
        ):

            return int(

                detection_group.loc[

                    detection_group[
                        "class_id"
                    ]
                    ==
                    class_id,

                    "track_id"

                ].nunique()

            )


        car_unique = (
            count_class(
                CAR
            )
        )

        motorcycle_unique = (
            count_class(
                MOTORCYCLE
            )
        )

        bus_unique = (
            count_class(
                BUS
            )
        )

        truck_unique = (
            count_class(
                TRUCK
            )
        )


    stopped_info = (

        track_window[

            track_window[
                "window_start_sec"
            ]
            ==
            start

        ]

    )


    stopped_count = int(

        stopped_info[
            "potentially_stopped"
        ].sum()

    )


    stopped_ratio = (

        stopped_count
        / unique_vehicles

        if unique_vehicles

        else 0.0

    )


    feature_rows.append({

        "window_start_sec":
            start,

        "window_end_sec":
            float(end),

        "unique_vehicle_count":
            unique_vehicles,

        "mean_vehicle_count_per_frame":
            mean_vehicle_count,

        "mean_normalized_speed":
            mean_speed,

        "median_normalized_speed":
            median_speed,

        "stopped_vehicle_count":
            stopped_count,

        "stopped_ratio":
            stopped_ratio,

        "bbox_coverage_proxy":
            coverage,

        "max_bbox_coverage_proxy":
            max_coverage,

        "mean_detection_confidence":
            mean_confidence,

        "car_unique":
            car_unique,

        "motorcycle_unique":
            motorcycle_unique,

        "bus_unique":
            bus_unique,

        "truck_unique":
            truck_unique
    })


external_data = pd.DataFrame(
    feature_rows
)


print(
    "\n✅ Basic features extracted"
)

print(
    "5-second windows:",
    len(
        external_data
    )
)


# ============================================================
# 10. TEMPORAL FEATURES
# ============================================================

TEMPORAL_COLUMNS = [

    "unique_vehicle_count",

    "mean_vehicle_count_per_frame",

    "mean_normalized_speed",

    "median_normalized_speed",

    "stopped_vehicle_count",

    "stopped_ratio",

    "bbox_coverage_proxy"
]


for column in TEMPORAL_COLUMNS:

    # Change from the previous 5-second window
    external_data[
        f"delta_{column}"
    ] = (

        external_data[
            column
        ]
        .diff()
        .fillna(0)

    )


    # Mean of current + previous two windows
    external_data[
        f"roll3_{column}"
    ] = (

        external_data[
            column
        ]
        .rolling(
            window=3,
            min_periods=1
        )
        .mean()

    )


# ============================================================
# 11. CHECK MODEL FEATURES
# ============================================================

missing_features = [

    feature

    for feature
    in incident_features

    if feature
    not in external_data.columns
]


if missing_features:

    raise RuntimeError(

        "Missing model features: "
        + str(
            missing_features
        )

    )


# ============================================================
# 12. RANDOM FOREST PREDICTIONS
# ============================================================

X_external = (

    external_data[
        incident_features
    ]

)


external_data[
    "incident_probability"
] = (

    incident_model
    .predict_proba(
        X_external
    )[:, 1]

)


external_data[
    "raw_prediction"
] = (

    external_data[
        "incident_probability"
    ]

    >=

    incident_threshold

).astype(int)


# ============================================================
# 13. TEMPORAL CONFIRMATION
# ============================================================

probabilities = (

    external_data[
        "incident_probability"
    ]
    .to_numpy()

)


confirmed_predictions = []


for i in range(
    len(
        probabilities
    )
):

    recent = (

        probabilities[

            max(
                0,
                i - 2
            ):
            i + 1

        ]

    )


    if temporal_rule == "raw":

        confirmed = bool(

            recent[-1]
            >=
            incident_threshold

        )


    elif temporal_rule == "two_consecutive":

        raw_recent = (

            recent
            >=
            incident_threshold

        ).astype(int)


        confirmed = (

            len(
                raw_recent
            )
            >=
            2

            and

            bool(
                raw_recent[-1]
                and
                raw_recent[-2]
            )

        )


    elif temporal_rule == "two_of_three":

        raw_recent = (

            recent
            >=
            incident_threshold

        ).astype(int)


        confirmed = bool(

            raw_recent.sum()
            >=
            2

        )


    elif temporal_rule == "mean3":

        confirmed = bool(

            recent.mean()
            >=
            incident_threshold

        )


    else:

        raise ValueError(

            f"Unknown temporal rule: "
            f"{temporal_rule}"

        )


    confirmed_predictions.append(
        int(
            confirmed
        )
    )


external_data[
    "confirmed_incident"
] = confirmed_predictions


external_data[
    "status"
] = np.where(

    external_data[
        "confirmed_incident"
    ]
    ==
    1,

    "INCIDENT",

    "NORMAL"
)


# ============================================================
# 14. FIND MOST SUSPICIOUS WINDOW
# ============================================================

MOST_SUSPICIOUS_ROW = (

    external_data
    .sort_values(

        "incident_probability",

        ascending=False

    )
    .iloc[0]

)


SUSPICIOUS_TIME = float(

    MOST_SUSPICIOUS_ROW[
        "window_start_sec"
    ]

)


SUSPICIOUS_PROBABILITY = float(

    MOST_SUSPICIOUS_ROW[
        "incident_probability"
    ]

)


SUSPICIOUS_STATUS = (

    MOST_SUSPICIOUS_ROW[
        "status"
    ]

)


print(
    "\n"
    + "=" * 60
)

print(
    "🚨 ML PIPELINE RESULT"
)

print(
    "=" * 60
)


print(
    "Most suspicious time:",
    SUSPICIOUS_TIME,
    "seconds"
)


print(
    "Probability:",
    round(
        SUSPICIOUS_PROBABILITY,
        4
    )
)


print(
    "Temporal verdict:",
    SUSPICIOUS_STATUS
)


# Show all incident windows
incident_windows = (

    external_data[

        external_data[
            "confirmed_incident"
        ]
        ==
        1

    ]

)


print(
    "Confirmed incident windows:",
    len(
        incident_windows
    )
)


if not incident_windows.empty:

    display(

        incident_windows[
            [
                "window_start_sec",
                "window_end_sec",
                "incident_probability",
                "unique_vehicle_count",
                "median_normalized_speed",
                "stopped_vehicle_count",
                "stopped_ratio"
            ]
        ]

    )


# ============================================================
# 15. SAVE EXTERNAL RESULTS
# ============================================================

OUTPUT_FILE = (

    PROJECT_ROOT
    / "outputs"
    / "external_video_predictions.csv"

)


OUTPUT_FILE.parent.mkdir(

    parents=True,

    exist_ok=True

)


external_data.to_csv(

    OUTPUT_FILE,

    index=False

)


print(
    "\n✅ Predictions saved to:"
)

print(
    OUTPUT_FILE
)


# ============================================================
# 16. LANGCHAIN TOOL FOR THIS UPLOADED VIDEO
# ============================================================

@tool
def analyze_uploaded_traffic_video(
    window_start_sec: float
) -> str:
    """
    Analyze the currently uploaded external traffic video
    near a requested five-second window.
    """

    nearest_index = (

        external_data[
            "window_start_sec"
        ]

        .sub(
            float(
                window_start_sec
            )
        )

        .abs()

        .idxmin()

    )


    position = (

        external_data
        .index
        .get_loc(
            nearest_index
        )

    )


    start_position = max(

        0,

        position - 2

    )


    recent = (

        external_data
        .iloc[
            start_position:
            position + 1
        ]

    )


    current = (
        recent.iloc[-1]
    )


    result = {

        "window_start_sec":
            float(
                current[
                    "window_start_sec"
                ]
            ),

        "window_end_sec":
            float(
                current[
                    "window_end_sec"
                ]
            ),

        "model":
            artifact[
                "model_name"
            ],

        "threshold":
            incident_threshold,

        "temporal_rule":
            temporal_rule,

        "current_probability":
            round(

                float(
                    current[
                        "incident_probability"
                    ]
                ),

                4

            ),

        "recent_probabilities":
            [

                round(
                    float(p),
                    4
                )

                for p
                in recent[
                    "incident_probability"
                ]

            ],

        "detector_verdict":
            str(
                current[
                    "status"
                ]
            ),

        "traffic_evidence": {

            "unique_vehicle_count":
                int(
                    current[
                        "unique_vehicle_count"
                    ]
                ),

            "mean_vehicle_count_per_frame":
                round(

                    float(
                        current[
                            "mean_vehicle_count_per_frame"
                        ]
                    ),

                    3

                ),

            "median_normalized_speed":
                round(

                    float(
                        current[
                            "median_normalized_speed"
                        ]
                    ),

                    3

                ),

            "stopped_vehicle_count":
                int(
                    current[
                        "stopped_vehicle_count"
                    ]
                ),

            "stopped_ratio":
                round(

                    float(
                        current[
                            "stopped_ratio"
                        ]
                    ),

                    3

                ),

            "bbox_coverage_proxy":
                round(

                    float(
                        current[
                            "bbox_coverage_proxy"
                        ]
                    ),

                    4

                )
        }
    }


    return json.dumps(
        result,
        indent=2
    )


# ============================================================
# 17. CONNECT APINEX + LANGCHAIN
# ============================================================

APINEX_API_KEY = userdata.get(
    "APINEX_API_KEY"
)


if not APINEX_API_KEY:

    raise ValueError(
        "APINEX_API_KEY is missing from Colab Secrets."
    )


llm = ChatOpenAI(

    model=
        "free/gpt-6-luna",

    api_key=
        APINEX_API_KEY,

    base_url=
        "https://api.apinex.bond/v1",

    temperature=0

)


SYSTEM_PROMPT_EXTERNAL = """
You are the Incident Detection Agent of an intelligent
multi-agent traffic-management system.

You MUST use the analyze_uploaded_traffic_video tool
before making a decision.

The tool uses:
- YOLO vehicle detection
- ByteTrack vehicle tracking
- 5-second traffic features
- temporal features
- a trained Random Forest classifier
- temporal confirmation

Rules:

1. The detector_verdict is the final detector decision.

2. current_probability is the current Random Forest
incident probability.

3. recent_probabilities contain recent five-second
incident probabilities.

4. YOLO detects vehicles only.
Do not say YOLO detects accidents.

5. normalized speed is an image-motion proxy.
It is not km/h.

6. unique_vehicle_count represents unique tracked
vehicle IDs in the window, not necessarily vehicles
present simultaneously.

7. stopped_vehicle_count is a heuristic.

8. Do not invent the exact incident type.

9. There is no trained severity classifier.
Always report severity as UNKNOWN.

10. If INCIDENT is confirmed:
recommend notifying the Traffic Signal Agent
and continuing monitoring.

11. If NORMAL:
recommend normal monitoring.

Return:

STATUS:
SEVERITY:
INCIDENT_PROBABILITY:
TEMPORAL_EVIDENCE:
TRAFFIC_EVIDENCE:
RECOMMENDED_ACTION:
"""


external_agent = create_agent(

    model=llm,

    tools=[
        analyze_uploaded_traffic_video
    ],

    system_prompt=
        SYSTEM_PROMPT_EXTERNAL

)


print(
    "\n✅ LangChain external-video agent ready"
)


# ============================================================
# 18. ASK AGENT ABOUT MOST SUSPICIOUS MOMENT
# ============================================================

print(
    "\n🤖 Asking Agent to analyze "
    "the most suspicious moment..."
)


try:

    response = external_agent.invoke({

        "messages": [{

            "role":
                "user",

            "content":
                (
                    f"Analyze the uploaded traffic video "
                    f"at {SUSPICIOUS_TIME} seconds. "
                    "Use the trained detector and determine "
                    "whether an incident is confirmed."
                )

        }]

    })


    print(
        "\n"
        + "=" * 60
    )

    print(
        "🤖 FINAL INCIDENT AGENT RESPONSE"
    )

    print(
        "=" * 60
    )


    print(

        response[
            "messages"
        ][-1].content

    )


except Exception as e:

    print(
        "\n⚠️ ML pipeline completed successfully, "
        "but the LLM call failed."
    )

    print(
        "API error:",
        e
    )

    print(
        "\nYou can still use the ML result above."
    )

In [ ]:
from pathlib import Path

PROJECT_ROOT = Path("/content/drive/MyDrive/5piproject")

MODEL_FILE = PROJECT_ROOT / "models/incident_detector.joblib"
FEATURE_FILE = (
    PROJECT_ROOT /
    "data/processed/all_traffic_features_temporal.csv"
)

print("Model exists:", MODEL_FILE.exists())
print("Features exist:", FEATURE_FILE.exists())

print("\nModel size:")
if MODEL_FILE.exists():
    print(round(MODEL_FILE.stat().st_size / 1024**2, 2), "MB")

print("\nAvailable notebooks:")

for folder in [
    Path("/content/drive/MyDrive/Colab Notebooks"),
    PROJECT_ROOT
]:
    if folder.exists():
        for notebook in folder.rglob("*.ipynb"):
            if "incident" in notebook.name.lower():
                print(notebook)